In [ ]:
# Core
import os
import sys

!{sys.executable} -m pip install numpy pandas matplotlib scikit-learn
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Modeling
from dataclasses import dataclass
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Business costs
FN_COST = 5000
FP_COST = 200

In [ ]:
# Project paths (assuming notebook in /notebooks)
DATA_PATH = "../data/customer_churn_1m.csv"
OUTPUT_DIR = "../outputs"
METRICS_DIR = os.path.join(OUTPUT_DIR, "metrics")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")

for d in [OUTPUT_DIR, METRICS_DIR, PLOTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# Update these if your dataset uses different names
TARGET_COL = "churn"
DATE_COL = "signup_date"

print("DATA_PATH:", DATA_PATH)
print("Output folders created.")

In [ ]:
## 1) Load Data and Quick Inspection

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.head(20))

In [ ]:
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in df.select_dtypes(include=["object"]).columns:
        ratio = df[col].nunique(dropna=True) / max(len(df), 1)
        if ratio < 0.5:
            df[col] = df[col].astype("category")
    return df

df = optimize_dtypes(df)
print("Memory-optimized dtypes applied.")

In [ ]:
print("Target distribution (%):")
display((df[TARGET_COL].value_counts(normalize=True) * 100).rename("percent"))

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("\nTop missing columns (%):")
display(missing_pct.head(15))

In [ ]:
## 2) Feature Engineering + Outlier Handling
- Parse `signup_date`
- Create date-derived features
- Cap numeric outliers (IQR-based, vectorized)

In [ ]:
def add_date_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df["signup_year"] = df[date_col].dt.year
        df["signup_month"] = df[date_col].dt.month
        df["signup_quarter"] = df[date_col].dt.quarter
        df["signup_dayofweek"] = df[date_col].dt.dayofweek
    return df

df = add_date_features(df, DATE_COL)
display(df[[DATE_COL, "signup_year", "signup_month", "signup_quarter", "signup_dayofweek"]].head())

In [ ]:
def cap_outliers_iqr(df: pd.DataFrame, numeric_cols: list, iqr_mult: float = 1.5) -> pd.DataFrame:
    q1 = df[numeric_cols].quantile(0.25)
    q3 = df[numeric_cols].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - iqr_mult * iqr
    upper = q3 + iqr_mult * iqr
    df[numeric_cols] = df[numeric_cols].clip(lower=lower, upper=upper, axis=1)
    return df

num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COL in num_cols_all:
    num_cols_all.remove(TARGET_COL)

df = cap_outliers_iqr(df, num_cols_all, iqr_mult=1.5)
print("Outlier capping complete.")

In [ ]:
## 3) Validation Strategy (Time-Aware)
To avoid temporal leakage, we split by time:
- Train: earliest 80%
- Validation: latest 20%

This simulates production behavior (predicting future churn from past data).

In [ ]:
# Sort by signup date first
df = df.sort_values(DATE_COL).reset_index(drop=True)

# Drop raw date after sort (keep engineered date features)
df_model = df.drop(columns=[DATE_COL])

split_idx = int(len(df_model) * 0.8)
train_df = df_model.iloc[:split_idx].copy()
val_df = df_model.iloc[split_idx:].copy()

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL].astype(int)
X_val = val_df.drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL].astype(int)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)

In [ ]:
## 4) Preprocessing + Modeling Pipelines
- Numeric: median imputation + scaling  
- Categorical: mode imputation + one-hot encoding  
- Models:
  1. Logistic Regression (baseline)
  2. HistGradientBoosting (strong nonlinear benchmark)

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

logit_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
])

hgb_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        max_depth=8,
        learning_rate=0.05,
        max_iter=250,
        random_state=RANDOM_STATE
    ))
])

In [ ]:
logit_model.fit(X_train, y_train)
hgb_model.fit(X_train, y_train)

val_prob_logit = logit_model.predict_proba(X_val)[:, 1]
val_prob_hgb = hgb_model.predict_proba(X_val)[:, 1]

def model_metrics(y_true, y_prob):
    return {
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob)
    }

m_logit = model_metrics(y_val, val_prob_logit)
m_hgb = model_metrics(y_val, val_prob_hgb)

metrics_df = pd.DataFrame([m_logit, m_hgb], index=["Logistic", "HistGB"])
display(metrics_df)

In [ ]:
# Select best model by PR-AUC (more useful in churn imbalance contexts)
if m_hgb["PR_AUC"] >= m_logit["PR_AUC"]:
    best_name = "HistGB"
    best_model = hgb_model
    best_prob = val_prob_hgb
else:
    best_name = "Logistic"
    best_model = logit_model
    best_prob = val_prob_logit

print("Selected model:", best_name)

In [ ]:
## 5) Cost-Based Threshold Optimization
We evaluate thresholds from 0.01 to 0.99 and choose the threshold that minimizes:

\[
\text{Total Cost} = (FN \times 5000) + (FP \times 200)
\]

In [ ]:
@dataclass
class CostResult:
    threshold: float
    tn: int
    fp: int
    fn: int
    tp: int
    total_cost: float
    precision: float
    recall: float

def evaluate_at_threshold(y_true, y_prob, threshold, fn_cost=FN_COST, fp_cost=FP_COST):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = fn * fn_cost + fp * fp_cost
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return CostResult(threshold, tn, fp, fn, tp, total_cost, precision, recall)

def find_optimal_threshold(y_true, y_prob, step=0.01):
    rows = []
    for t in np.arange(step, 1.0, step):
        res = evaluate_at_threshold(y_true, y_prob, t)
        rows.append({
            "threshold": res.threshold,
            "tn": res.tn, "fp": res.fp, "fn": res.fn, "tp": res.tp,
            "precision": res.precision,
            "recall": res.recall,
            "total_cost": res.total_cost
        })
    table = pd.DataFrame(rows).sort_values("total_cost", ascending=True).reset_index(drop=True)
    return table, table.loc[0, "threshold"]

In [ ]:
cost_table, best_threshold = find_optimal_threshold(y_val.values, best_prob, step=0.01)

default_eval = evaluate_at_threshold(y_val.values, best_prob, 0.50)
optimal_eval = evaluate_at_threshold(y_val.values, best_prob, best_threshold)

print(f"Best threshold: {best_threshold:.2f}")
print(f"Cost at threshold 0.50: ${default_eval.total_cost:,.0f}")
print(f"Cost at threshold {best_threshold:.2f}: ${optimal_eval.total_cost:,.0f}")
print(f"Estimated savings: ${default_eval.total_cost - optimal_eval.total_cost:,.0f}")

display(cost_table.head(10))

In [ ]:
y_pred_opt = (best_prob >= best_threshold).astype(int)

print("Confusion Matrix (Optimal Threshold):")
print(confusion_matrix(y_val, y_pred_opt))

print("\nClassification Report (Optimal Threshold):")
print(classification_report(y_val, y_pred_opt, digits=4))

In [ ]:
# ROC
fpr, tpr, _ = roc_curve(y_val, best_prob)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"{best_name} ROC")
plt.plot([0,1], [0,1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "roc_curve.png"), dpi=150)
plt.show()

# PR
precision, recall, _ = precision_recall_curve(y_val, best_prob)
plt.figure(figsize=(6,4))
plt.plot(recall, precision, label=f"{best_name} PR")
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "pr_curve.png"), dpi=150)
plt.show()

# Cost vs Threshold
plt.figure(figsize=(7,4))
plt.plot(cost_table["threshold"], cost_table["total_cost"])
plt.axvline(best_threshold, color="red", linestyle="--", label=f"Best={best_threshold:.2f}")
plt.title("Business Cost vs Decision Threshold")
plt.xlabel("Threshold")
plt.ylabel("Total Cost ($)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "cost_vs_threshold.png"), dpi=150)
plt.show()

In [ ]:
# Save metric tables
metrics_df.to_csv(os.path.join(METRICS_DIR, "model_probability_metrics.csv"), index=True)
cost_table.to_csv(os.path.join(METRICS_DIR, "threshold_cost_table.csv"), index=False)

summary = pd.DataFrame([{
    "selected_model": best_name,
    "optimal_threshold": round(float(best_threshold), 4),
    "default_cost": float(default_eval.total_cost),
    "optimal_cost": float(optimal_eval.total_cost),
    "estimated_savings": float(default_eval.total_cost - optimal_eval.total_cost),
    "fn_cost": FN_COST,
    "fp_cost": FP_COST
}])
summary.to_csv(os.path.join(METRICS_DIR, "business_summary.csv"), index=False)

print("Saved metrics and summary to:", METRICS_DIR)

In [ ]:
## Task 2: Validation Strategy (Answer)
I used a time-based holdout validation approach rather than a random split. Since churn is predicted after signup over time, training on older customers and validating on newer customers better reflects real deployment conditions and reduces the risk of temporal leakage. This approach gives a more realistic estimate of future business performance.